# 61. Pygame Games — `pygame`
**Python Libraries Lab · Phase 3 · GUI, Games & Multimedia**

_2D games: a game loop, sprites, input and collisions._

⚙️ **Runs in Colab** after the setup cell below.

Pygame is a library for making 2D games. Every game is built around a game loop that runs about sixty times a second. Each time round, it reads input from the keyboard and mouse, updates the positions of everything, checks for collisions, draws the new frame, and waits a moment. Once you understand that loop, you can build almost any 2D game.

**In this notebook:** You will build ShopEasy Catch, a game where you move a shopping basket left and right to catch falling products and avoid broken boxes. In Colab you will run the game logic with a robot player and see the frames as pictures. Then you will get the full playable version for your laptop.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Games need a screen and a keyboard, so the playable game runs on your laptop. In Colab, Pygame runs with a dummy display: you can draw frames, run the game logic and watch it as images.

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q pygame
import importlib.metadata as md
for pkg in ['pygame']:
    print(pkg, md.version(pkg))

## Step 1 · Pygame without a screen
The dummy driver lets us draw in Colab. A Surface is a picture you draw on.

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"
import pygame, random
import matplotlib.pyplot as plt
pygame.init()
W, H = 480, 360
screen = pygame.display.set_mode((W, H))

def show(surface, title=""):
    arr = pygame.surfarray.array3d(surface).swapaxes(0, 1)
    plt.figure(figsize=(5, 3.8)); plt.imshow(arr); plt.axis("off"); plt.title(title); plt.show()

screen.fill((235, 242, 250))
pygame.draw.rect(screen, (44, 90, 154), pygame.Rect(200, 320, 80, 24), border_radius=6)     # basket
pygame.draw.circle(screen, (224, 169, 62), (120, 80), 14)                                   # product
pygame.draw.rect(screen, (190, 60, 60), pygame.Rect(330, 140, 24, 24))                       # broken box
font = pygame.font.Font(None, 30)
screen.blit(font.render("ShopEasy Catch", True, (20, 26, 35)), (10, 10))
show(screen, "Shapes and text on a Surface")

## Step 2 · Game objects and collisions
Each falling thing has a Rect. `colliderect` checks if it touched the basket.

In [ ]:
class Game:
    def __init__(self, seed=1):
        self.rng = random.Random(seed)
        self.basket = pygame.Rect(W // 2 - 40, H - 40, 80, 24)
        self.items = []            # each: [rect, speed, is_bad]
        self.score, self.lives, self.frame = 0, 3, 0

    def spawn(self):
        x = self.rng.randint(10, W - 30)
        self.items.append([pygame.Rect(x, -20, 22, 22), self.rng.uniform(3, 6), self.rng.random() < 0.25])

    def update(self, move):            # move: -1 left, 0 stay, +1 right
        self.frame += 1
        self.basket.x = max(0, min(W - self.basket.width, self.basket.x + move * 8))
        if self.frame % 20 == 0:
            self.spawn()
        for item in self.items[:]:
            item[0].y += item[1]
            if item[0].colliderect(self.basket):
                if item[2]: self.lives -= 1
                else: self.score += 10
                self.items.remove(item)
            elif item[0].top > H:
                self.items.remove(item)

    def draw(self, surf):
        surf.fill((235, 242, 250))
        pygame.draw.rect(surf, (44, 90, 154), self.basket, border_radius=6)
        for rect, _, bad in self.items:
            if bad: pygame.draw.rect(surf, (190, 60, 60), rect)
            else: pygame.draw.circle(surf, (224, 169, 62), rect.center, 11)
        surf.blit(font.render(f"Score {self.score}   Lives {self.lives}", True, (20, 26, 35)), (10, 10))

print("Game class ready")

## Step 3 · Run the game loop with a robot player
The robot moves towards the nearest good item. We save three frames.

In [ ]:
def robot(game):
    good = [r for r, _, bad in game.items if not bad]
    if not good:
        return 0
    target = max(good, key=lambda r: r.y).centerx
    return (target > game.basket.centerx + 5) - (target < game.basket.centerx - 5)

game = Game()
clock = pygame.time.Clock()
snapshots = []
for frame in range(600):                        # 10 seconds at 60 FPS
    game.update(robot(game))
    if frame in (100, 300, 599):
        game.draw(screen); snapshots.append(screen.copy())
print("Score:", game.score, "| lives left:", game.lives)
for i, s in enumerate(snapshots):
    show(s, f"Frame {[100, 300, 599][i]}")

## Step 4 · The playable game for your laptop
Same `Game` idea plus keyboard input and a real window at 60 frames per second.

In [ ]:
%%writefile shopeasy_catch.py
import random, pygame

W, H = 480, 360
pygame.init()
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("ShopEasy Catch - arrows to move, R to restart")
font = pygame.font.Font(None, 30)
clock = pygame.time.Clock()

def new_game():
    return {"basket": pygame.Rect(W // 2 - 40, H - 40, 80, 24), "items": [], "score": 0, "lives": 3, "frame": 0}

g = new_game()
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN and event.key == pygame.K_r:
            g = new_game()
    keys = pygame.key.get_pressed()
    if g["lives"] > 0:
        g["basket"].x += (keys[pygame.K_RIGHT] - keys[pygame.K_LEFT]) * 8
        g["basket"].clamp_ip(screen.get_rect())
        g["frame"] += 1
        if g["frame"] % max(8, 25 - g["score"] // 50) == 0:
            g["items"].append([pygame.Rect(random.randint(10, W - 30), -20, 22, 22), random.uniform(3, 6) + g["score"] / 200, random.random() < 0.25])
        for item in g["items"][:]:
            item[0].y += item[1]
            if item[0].colliderect(g["basket"]):
                g["lives" if item[2] else "score"] += -1 if item[2] else 10
                g["items"].remove(item)
            elif item[0].top > H:
                g["items"].remove(item)
    screen.fill((235, 242, 250))
    pygame.draw.rect(screen, (44, 90, 154), g["basket"], border_radius=6)
    for rect, _, bad in g["items"]:
        if bad: pygame.draw.rect(screen, (190, 60, 60), rect)
        else: pygame.draw.circle(screen, (224, 169, 62), rect.center, 11)
    screen.blit(font.render(f"Score {g['score']}   Lives {g['lives']}", True, (20, 26, 35)), (10, 10))
    if g["lives"] <= 0:
        screen.blit(font.render("Game over! Press R to play again", True, (190, 60, 60)), (70, H // 2))
    pygame.display.flip()
    clock.tick(60)
pygame.quit()

## Step 5 · Check the game file
Valid Python? Then download it and play on your laptop.

In [ ]:
import py_compile
py_compile.compile("shopeasy_catch.py", doraise=True)
print("shopeasy_catch.py is valid. Laptop: pip install pygame, then python shopeasy_catch.py")

## ✍️ Your turn
Two rectangles: `a = pygame.Rect(0, 0, 50, 50)` and `b = pygame.Rect(40, 40, 20, 20)`. Store whether they **collide** (True/False) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer is True, f"Expected True, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
a = pygame.Rect(0, 0, 50, 50)
b = pygame.Rect(40, 40, 20, 20)
answer = a.colliderect(b)

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Surface**
- **Game loop**
- **Event**
- **Rect / collision**
- **Frame rate (FPS)**
- **Sprite**

**Next:** 62. Kivy Touch Apps (`kivy`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/62_kivy.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install pygame
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.
4. **Run the app:** download `shopeasy_catch.py` (Colab: Files panel on the left → right-click → Download) into the `python-lab` folder, then run `python shopeasy_catch.py` in the same terminal.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.